In [ ]:
import os
import shutil
import json
import time
from collections import Counter, defaultdict, deque

# =========================================================
# 1. SETUP & PATHS
# =========================================================
print("Cleaning workspace for dataset creation...")
%cd /kaggle/working
!rm -rf 26LPCVC_Track2_Sample_Solution qevd_final /tmp/qevd_staging /tmp/qevd_raw

REPO_DIR = '/kaggle/working/26LPCVC_Track2_Sample_Solution'
STAGING_DIR = '/tmp/qevd_staging'
RAW_EXTRACT_DIR = '/tmp/qevd_raw'
FINAL_DATASET_DIR = '/kaggle/working/qevd_final'

# Soft cap for headroom + hard cap safeguard near Kaggle limit.
MAX_WORKING_GB = 18.0
MAX_WORKING_BYTES = int(MAX_WORKING_GB * 1024 ** 3)

MOVE_RETRIES = 3
CLASS_SUMMARY_PATH = os.path.join(FINAL_DATASET_DIR, 'class_selection_summary.json')

os.makedirs(STAGING_DIR, exist_ok=True)
os.makedirs(RAW_EXTRACT_DIR, exist_ok=True)
os.makedirs(FINAL_DATASET_DIR, exist_ok=True)



In [ ]:
# =========================================================
# 2. CLONE SAMPLE REPO ONCE (for labels + scripts)
# =========================================================
print("Cloning sample solution repo...")
!git clone https://github.com/lpcvai/26LPCVC_Track2_Sample_Solution.git {REPO_DIR}

LABEL_JSON = os.path.join(REPO_DIR, 'fine_grained_labels_release.json')
if not os.path.exists(LABEL_JSON):
    raise FileNotFoundError(f"Label file missing: {LABEL_JSON}")

In [ ]:
# =========================================================
# 3. DOWNLOAD + EXTRACT QEVD PART-2
# =========================================================
print("Downloading QEVD Part 2 archives...")
BASE_URL = "https://softwarecenter.qualcomm.com/api/download/software/dataset/AIDataset/Qualcomm_Exercise_Video_Dataset/QEVD-FIT-300k-Part-2"
!wget -q -O {STAGING_DIR}/p2.zip "{BASE_URL}/QEVD-FIT-300k-Part-2.zip"
!wget -q -O {STAGING_DIR}/p2.z01 "{BASE_URL}/QEVD-FIT-300k-Part-2.z01"
!wget -q -O {STAGING_DIR}/p2.z02 "{BASE_URL}/QEVD-FIT-300k-Part-2.z02"

print("Extracting split archive...")
!7z x {STAGING_DIR}/p2.zip -o{RAW_EXTRACT_DIR} -y > /dev/null

In [ ]:
# =========================================================
# 4. ORGANIZE VIDEOS INTO CLASS FOLDERS + COVERAGE CHECKS
# =========================================================
print("Organizing videos by class label...")
with open(LABEL_JSON, 'r', encoding='utf-8') as f:
    labels_data = json.load(f)

raw_part_dir = os.path.join(RAW_EXTRACT_DIR, 'QEVD-FIT-300k-Part-2')
if not os.path.isdir(raw_part_dir):
    raise FileNotFoundError(f"Extracted folder not found: {raw_part_dir}")

# Build expected class coverage from labels metadata.
def normalize_label(label: str) -> str:
    return label.strip().replace(' ', '_').replace('/', '-')

# Enumerate extracted files once.
all_source_files = [
    os.path.join(raw_part_dir, fn)
    for fn in os.listdir(raw_part_dir)
    if os.path.isfile(os.path.join(raw_part_dir, fn))
]
source_total_bytes = sum(os.path.getsize(p) for p in all_source_files)
print(f"Extracted source size: {source_total_bytes / (1024**3):.2f} GB")
print(f"Soft budget cap:      {MAX_WORKING_BYTES / (1024**3):.2f} GB")

# Index extracted files by basename to avoid repeated os.path.exists calls.
source_index = {os.path.basename(p): p for p in all_source_files}

In [ ]:
# Build per-class candidate pool from labels that are actually present in this part.
available_by_class = defaultdict(list)
missing = 0
for entry in labels_data:
    v_path = entry.get('video_path', '')
    filename = os.path.basename(v_path)
    src = source_index.get(filename)
    if src is None:
        missing += 1
        continue

    cls = normalize_label(entry.get('label', 'unknown'))
    available_by_class[cls].append(src)

if not available_by_class:
    sample_files = sorted([os.path.basename(p) for p in all_source_files[:10]])
    raise RuntimeError(
        "No labeled videos matched extracted Part-2 files. "
        f"Extracted files={len(all_source_files)}, labels={len(labels_data)}. "
        f"Example extracted names: {sample_files}"
    )

print(f"Classes with available videos in Part-2: {len(available_by_class)}")
print(f"Label entries missing in this part:      {missing}")

# Ensure every observed class directory exists.
for cls_name in available_by_class.keys():
    os.makedirs(os.path.join(FINAL_DATASET_DIR, cls_name), exist_ok=True)


In [ ]:
# -------- Storage-aware class-balanced selector (round robin) --------
# 1) Reserve one smallest file per class when it fits in budget.
selected_files = []
selected_by_class = Counter()
selected_paths = set()
used_bytes = 0
skipped_classes_due_budget = []

for cls in sorted(available_by_class.keys()):
    cls_sorted = sorted(available_by_class[cls], key=lambda p: os.path.getsize(p))
    first_path = cls_sorted[0]
    first_size = os.path.getsize(first_path)
    if used_bytes + first_size <= MAX_WORKING_BYTES:
        selected_files.append((cls, first_path, first_size))
        selected_by_class[cls] += 1
        selected_paths.add(first_path)
        used_bytes += first_size
    else:
        skipped_classes_due_budget.append(cls)

if skipped_classes_due_budget:
    print(
        f"Warning: skipped {len(skipped_classes_due_budget)} classes due to strict {MAX_WORKING_GB:.1f} GB cap. "
        f"Examples: {skipped_classes_due_budget[:10]}"
    )

In [ ]:
# 2) Add more files by round robin across classes until budget is full.
rr_queues = {}
for cls in sorted(available_by_class.keys()):
    remaining = [
        p for p in sorted(available_by_class[cls], key=lambda p: os.path.getsize(p))
        if p not in selected_paths
    ]
    rr_queues[cls] = deque(remaining)

active_classes = deque([cls for cls in sorted(rr_queues.keys()) if rr_queues[cls]])
while active_classes:
    cls = active_classes.popleft()
    if not rr_queues[cls]:
        continue

    candidate = rr_queues[cls].popleft()
    csize = os.path.getsize(candidate)
    if used_bytes + csize <= MAX_WORKING_BYTES:
        selected_files.append((cls, candidate, csize))
        selected_by_class[cls] += 1
        used_bytes += csize

    if rr_queues[cls]:
        active_classes.append(cls)

print(f"Selected subset size: {used_bytes / (1024**3):.2f} GB")
print(f"Selected videos:      {len(selected_files)}")
print(f"Covered classes:      {len(selected_by_class)}")

In [ ]:
# Safe move to reduce transient move failures.
def safe_move(src: str, dst: str) -> None:
    for _ in range(MOVE_RETRIES):
        try:
            shutil.move(src, dst)
            return
        except Exception:
            time.sleep(0.2)
    # Fallback path for cross-device / transient move issues.
    shutil.copy2(src, dst)
    os.remove(src)

In [ ]:
# Move selected files into class folders.
moved_by_class = Counter()
moved = 0
move_failures = []
for cls, src, _ in selected_files:
    filename = os.path.basename(src)
    dst_dir = os.path.join(FINAL_DATASET_DIR, cls)
    dst_path = os.path.join(dst_dir, filename)

    try:
        safe_move(src, dst_path)
        moved += 1
        moved_by_class[cls] += 1
        if moved % 2000 == 0:
            print(f"Moved {moved} videos...")
    except Exception as e:
        if len(move_failures) < 20:
            move_failures.append({'src': src, 'dst': dst_path, 'error': str(e)})

In [ ]:
# Strict class coverage checks for selected subset.
classes_with_zero = sorted([c for c in selected_by_class.keys() if moved_by_class[c] == 0])
classes_mismatch = sorted([c for c in selected_by_class.keys() if moved_by_class[c] != selected_by_class[c]])

print(f"Done. moved={moved}, classes={len(selected_by_class)}")
if move_failures:
    print(f"Move failures captured: {len(move_failures)} (showing up to 20)")
    print(move_failures[:3])

if classes_with_zero:
    raise RuntimeError(
        f"Coverage failure: {len(classes_with_zero)} classes have zero moved videos. "
        f"Examples: {classes_with_zero[:10]}"
    )

if classes_mismatch:
    raise RuntimeError(
        f"Move mismatch in {len(classes_mismatch)} classes. "
        f"Examples: {classes_mismatch[:10]}"
    )

In [ ]:
# Persist class summary for downstream audit.
summary = {
    'budget_gb': round(MAX_WORKING_BYTES / (1024**3), 3),
    'selected_total': int(len(selected_files)),
    'moved_total': int(moved),
    'covered_classes': int(len(selected_by_class)),
    'skipped_classes_due_budget': skipped_classes_due_budget,
    'missing_label_entries': int(missing),
    'selected_by_class': dict(sorted(selected_by_class.items())),
    'moved_by_class': dict(sorted(moved_by_class.items())),
}
with open(CLASS_SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(f"Class summary written: {CLASS_SUMMARY_PATH}")

if skipped_classes_due_budget:
    print("Completed under strict cap with class drops due to budget (see summary JSON).")
else:
    print("Class coverage check passed: no class left behind in selected subset.")

In [ ]:
# =========================================================
# 5. CLEAN TEMP DATA (keep repo + final dataset)
# =========================================================
print("Cleaning temporary download/extract directories...")
!rm -rf {STAGING_DIR} {RAW_EXTRACT_DIR}

print("SUCCESS")
print(f"Final dataset directory: {FINAL_DATASET_DIR}")
print("Next: save this notebook output as a Kaggle Dataset.")